In [ ]:
"""loopback test with a readout tone"""
import sys
sys.path.insert(0, r"c:\Users\Joshi-A\Documents\HouckLab_QICK")
from qick.asm_v2 import AveragerProgramV2
from WorkingProjects.gates_team.CoreLib.socProxy import makeProxy
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
class LoopbackProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch = cfg["ro_ch"]
        gen_ch = cfg["gen_ch"]

        self.declare_gen(
            ch=gen_ch, nqz=cfg["nqz"], mixer_freq=cfg["mixer_freq"], ro_ch=ro_ch
        )

        self.declare_readout(ch=ro_ch, length=cfg["ro_len"])

        self.add_readoutconfig(
            ch=ro_ch, name="myro", freq=cfg["freq"], gen_ch=gen_ch, outsel="product"
        )

        self.add_cosine(
            ch=gen_ch, name="ramp", length=cfg["ramp_len"], even_length=True
        )

        self.add_pulse(
            ch=gen_ch,
            name="mypulse",
            ro_ch=ro_ch,
            style="const",
            # style="flat_top",
            # envelope="ramp",
            freq=cfg["freq"],
            length=cfg["flat_len"],
            phase=cfg["phase"],
            gain=cfg["gain"],
        )

        self.send_readoutconfig(ch=cfg["ro_ch"], name="myro", t=0)

    def _body(self, cfg):
        self.delay_auto()
        self.pulse(ch=cfg["gen_ch"], name="mypulse", t=0.0)
        self.trigger(ros=[cfg["ro_ch"]], pins=[0], t=cfg["trig_time"], mr=True)


In [ ]:
config = {
    "gen_ch": 8,
    "ro_ch": 10,
    "mixer_freq": 3500,
    "freq": 4000,
    "nqz": 2,
    "trig_time": 0.0,
    "ro_len": 3.0,
    "flat_len": 1.0,
    "ramp_len": 1.0,
    "phase": 0,
    "gain": 1.0,
}

soc, soccfg = makeProxy()
prog = LoopbackProgram(soccfg, reps=1, final_delay=0.5, cfg=config)
freq = config["freq"]

soc.rfb_set_gen_filter(config["gen_ch"], fc=freq / 1000, ftype="bandpass", bw=1.0)
soc.rfb_set_ro_filter(config["ro_ch"], fc=freq / 1000, ftype="bandpass", bw=1.0)

# Set attenuator on DAC.
soc.rfb_set_gen_rf(config["gen_ch"], 20, 20)
# Set attenuator on ADC.
soc.rfb_set_ro_rf(config["ro_ch"], 25)

# observe pulse envelope
iq_list = prog.acquire_decimated(soc, rounds=1)
t = prog.get_time_axis(ro_index=0)
env_I = iq_list[0][:, 0]
env_Q = iq_list[0][:, 1]
env_Mag = np.abs(iq_list[0].dot([1, 1j]))
plt.plot(t, env_I, label="I value")
plt.plot(t, env_Q, label="Q value")
plt.plot(t, env_Mag, label="Magnitude")
plt.legend()
plt.ylabel("amplitude [ADU]")
plt.xlabel("time [us]")
plt.show()

print(np.average(env_I[(t >=0.75) & (t <= 1.25)]))
print(np.average(env_Q[(t >= 0.75) & (t <= 1.25)]))
print(np.average(env_Mag[(t >= 0.75) & (t <= 1.25)]))
# observe time-domain waveform
# soc.arm_mr(ch=config["ro_ch"])
# iq_list = prog.acquire_decimated(soc, rounds=1)
# iq_mr = soc.get_mr()[:800]
# t = prog.get_time_axis_mr(config["ro_ch"], iq_mr)
# plt.plot(t, iq_mr[:, 0], label="I")
# plt.plot(t, iq_mr[:, 1], label="Q")
# plt.xlabel("us")
# plt.legend()
# plt.show()

## Sweep (Att1,Att2)

In [ ]:
config = {
    "gen_ch": 8,
    "ro_ch": 10,
    "mixer_freq": 3500,
    "freq": 4000,
    "nqz": 2,
    "trig_time": 0.0,
    "ro_len": 3.0,
    "flat_len": 1.0,
    "ramp_len": 1.0,
    "phase": 90,
    "gain": -1.0,
}

In [ ]:
import os
import contextlib
import numpy as np

dict = {"att1": [], "att2": [], "avg_I": [], "avg_Q": [], "avg_Mag": []}

with open(os.devnull, "w") as f:
    with contextlib.redirect_stdout(f), contextlib.redirect_stderr(f):
        soc, soccfg = makeProxy()

for i in range(1000):
    att_arr = np.array([10, 15, 20, 25, 30])
    att1 = np.random.choice(att_arr)
    att2 = np.random.choice(att_arr)
    dict['att1'].append(att1)
    dict["att2"].append(att2)

    # config = {
    #     "gen_ch": 8,
    #     "ro_ch": 10,
    #     "mixer_freq": 3500,
    #     "freq": 4000,
    #     "nqz": 1,
    #     "trig_time": 0.0,
    #     "ro_len": 3.0,
    #     "flat_len": 1.0,
    #     "ramp_len": 1.0,
    #     "phase": 90,
    #     "gain": 1.0,
    # }

    # soc, soccfg = makeProxy()
    prog = LoopbackProgram(soccfg, reps=1, final_delay=0.5, cfg=config)
    freq = config["freq"]

    soc.rfb_set_gen_filter(config["gen_ch"], fc=freq / 1000, ftype="bandpass", bw=1.0)
    soc.rfb_set_ro_filter(config["ro_ch"], fc=freq / 1000, ftype="bandpass", bw=1.0)

    # Set attenuator on DAC.
    soc.rfb_set_gen_rf(config["gen_ch"], att1, att2)

    # Set attenuator on ADC.
    soc.rfb_set_ro_rf(config["ro_ch"], 25)

    # observe pulse envelope
    with open(os.devnull, "w") as f:
        with contextlib.redirect_stdout(f), contextlib.redirect_stderr(f):
            iq_list = prog.acquire_decimated(soc, rounds=1)
    t = prog.get_time_axis(ro_index=0)
    env_I = iq_list[0][:, 0]
    env_Q = iq_list[0][:, 1]
    env_Mag = np.abs(iq_list[0].dot([1, 1j]))
    # plt.plot(t, env_I, label="I value")
    # plt.plot(t, env_Q, label="Q value")
    # plt.plot(t, env_Mag, label="Magnitude")
    # plt.legend()
    # plt.ylabel("amplitude [ADU]")
    # plt.xlabel("time [us]")
    # plt.show()

    dict["avg_I"].append(np.average(env_I[(t >= 0.75) & (t <= 1.25)]))
    dict["avg_Q"].append(np.average(env_Q[(t >= 0.75) & (t <= 1.25)]))
    dict["avg_Mag"].append(np.average(env_Mag[(t >= 0.75) & (t <= 1.25)]))


In [ ]:
plt.figure()
plt.plot(dict['att1'], np.abs(dict['avg_I']), '.', label='I')
plt.plot(dict["att1"], np.abs(dict["avg_Q"]), ".", label="Q")
plt.plot(dict["att1"], dict["avg_Mag"], ".", label="Mag")
plt.xlabel("Att1 (dB)")
plt.ylabel("Amplitude (a.u.)")
plt.legend(loc="center left", bbox_to_anchor=(1.05, 0.5))
plt.title("Pulse Amplitude vs. 1st Stage Attenuation")
plt.show()

In [ ]:
plt.figure()
plt.plot(dict["att2"], np.abs(dict["avg_I"]), ".", label="I")
plt.plot(dict["att2"], np.abs(dict["avg_Q"]), ".", label="Q")
plt.plot(dict["att2"], dict["avg_Mag"], ".", label="Mag")
plt.xlabel("Att2 (dB)")
plt.ylabel("Amplitude (a.u.)")
plt.legend(loc="center left", bbox_to_anchor=(1.05, 0.5))
plt.title("Pulse Amplitude vs. 2nd Stage Attenuation")
plt.show()


In [ ]:
# plt.figure()
# sc = plt.scatter(dict["att1"], dict["att2"], c=np.abs(dict["avg_I"]), cmap="viridis", s=2, alpha=1)
# plt.xlabel("att1")
# plt.ylabel("att2")
# plt.colorbar(sc, label="avg_I")
# plt.show()

## Sweep Att_ro

In [ ]:
import os
import contextlib
import numpy as np

dict = {"att1": [], "att2": [], "att_ro":[], "avg_I": [], "avg_Q": [], "avg_Mag": []}

with open(os.devnull, "w") as f:
    with contextlib.redirect_stdout(f), contextlib.redirect_stderr(f):
        soc, soccfg = makeProxy()

att_arr = np.array([21, 22, 23, 24, 25, 26, 27, 28, 29, 30])
for i in range(1000):
    att_ro = np.random.choice(att_arr)
    dict["att_ro"].append(att_ro)

    # config = {
    #     "gen_ch": 8,
    #     "ro_ch": 10,
    #     "mixer_freq": 3500,
    #     "freq": 4000,
    #     "nqz": 1,
    #     "trig_time": 0.0,
    #     "ro_len": 3.0,
    #     "flat_len": 1.0,
    #     "ramp_len": 1.0,
    #     "phase": 90,
    #     "gain": 1.0,
    # }

    # soc, soccfg = makeProxy()
    prog = LoopbackProgram(soccfg, reps=1, final_delay=0.5, cfg=config)
    freq = config["freq"]

    soc.rfb_set_gen_filter(config["gen_ch"], fc=freq / 1000, ftype="bandpass", bw=1.0)
    soc.rfb_set_ro_filter(config["ro_ch"], fc=freq / 1000, ftype="bandpass", bw=1.0)

    # Set attenuator on DAC.
    soc.rfb_set_gen_rf(config["gen_ch"], 30, 30)

    # Set attenuator on ADC.
    soc.rfb_set_ro_rf(config["ro_ch"], att_ro)

    # observe pulse envelope
    with open(os.devnull, "w") as f:
        with contextlib.redirect_stdout(f), contextlib.redirect_stderr(f):
            iq_list = prog.acquire_decimated(soc, rounds=1)
    t = prog.get_time_axis(ro_index=0)
    env_I = iq_list[0][:, 0]
    env_Q = iq_list[0][:, 1]
    env_Mag = np.abs(iq_list[0].dot([1, 1j]))
    # plt.plot(t, env_I, label="I value")
    # plt.plot(t, env_Q, label="Q value")
    # plt.plot(t, env_Mag, label="Magnitude")
    # plt.legend()
    # plt.ylabel("amplitude [ADU]")
    # plt.xlabel("time [us]")
    # plt.show()

    dict["avg_I"].append(np.average(env_I[(t >= 0.75) & (t <= 1.25)]))
    dict["avg_Q"].append(np.average(env_Q[(t >= 0.75) & (t <= 1.25)]))
    dict["avg_Mag"].append(np.average(env_Mag[(t >= 0.75) & (t <= 1.25)]))


In [ ]:
plt.figure()
plt.plot(dict["att_ro"], np.abs(dict["avg_I"]), ".", label="I")
plt.plot(dict["att_ro"], np.abs(dict["avg_Q"]), ".", label="Q")
plt.plot(dict["att_ro"], dict["avg_Mag"], ".", label="Mag")
plt.xlabel("Att_ro (dB)")
plt.ylabel("Amplitude (a.u.)")
plt.legend(loc="center left", bbox_to_anchor=(1.05, 0.5))
plt.title('Pulse Amplitude vs. Readout Attenuation')
plt.show()